### Settings

In [84]:
#imports
from pathlib import Path
import sys
sys.path.append('/Users/micahlim/Desktop/NMF_separation/NMFtoolbox/python')
import numpy as np
import librosa
from NMFtoolbox.NMFD import NMFD
from NMFtoolbox.NMF import NMF
from NMFtoolbox.utils import EPS
from mir_eval.separation import bss_eval_sources
import os
import matplotlib.pyplot as plt
import IPython.display as ipd
import soundfile as sf #library to read and write sound files
import statistics

In [92]:
#restart / delete all previous results

import shutil

for folder in ["NMFresults", "NMFDresults", "original_data_plot", "nmf_results_plot", "nmfd_results_plot"]:
    shutil.rmtree(folder, ignore_errors=True)
    os.makedirs(folder, exist_ok=True)

#create result folders
folders_nmf = [
    'NMFresults/static',
    'NMFresults/dynamic',
    'NMFresults/static_kick',
    'NMFresults/static_snare',
    'NMFresults/dynamic_kick',
    'NMFresults/dynamic_snare'
]

for folder in folders_nmf:
    os.makedirs(folder, exist_ok=True)


# NMFDresults folders
folders_nmfd = [
    'NMFDresults/static',
    'NMFDresults/dynamic',
    'NMFDresults/static_kick',
    'NMFDresults/static_snare',
    'NMFDresults/dynamic_kick',
    'NMFDresults/dynamic_snare'
]

for folder in folders_nmfd:
    os.makedirs(folder, exist_ok=True)

In [86]:
#parameters
sample_rate = 22050
FRAME = 512
HOP = 256
template_frames = 8
num_files = 1

### Plot Original Stems

In [87]:
def plot_stem_spectrograms(num_files, sample_rate, FRAME, HOP):
    """Plot spectrograms and magnitude/activations for each dataset stem."""
    rms_global_max = -np.inf  # will hold the global max (in dB) across all RMS envelopes and all files

    for i in range(num_files):
        static_kick, _ = librosa.load(f'dataset/static_kick/static_kick_{i:03d}.wav', sr=sample_rate)
        static_snare, _ = librosa.load(f'dataset/static_snare/static_snare_{i:03d}.wav', sr=sample_rate)
        dynamic_kick, _ = librosa.load(f'dataset/dynamic_kick/dynamic_kick_{i:03d}.wav', sr=sample_rate)
        dynamic_snare, _ = librosa.load(f'dataset/dynamic_snare/dynamic_snare_{i:03d}.wav', sr=sample_rate)

        stems = [
            ('Static Kick', static_kick),
            ('Static Snare', static_snare),
            ('Dynamic Kick', dynamic_kick),
            ('Dynamic Snare', dynamic_snare),
        ]

        # --- First pass: compute global RMS max (in dB) across all stems ---
        all_rms_db = []
        for label, audio in stems:
            stft = librosa.stft(audio, n_fft=FRAME, hop_length=HOP)
            mag = np.abs(stft)
            rms = np.sqrt(np.mean(mag ** 2, axis=0))
            rms_db = 20 * np.log10(rms + 1e-6)
            all_rms_db.append(rms_db)
        file_rms_db_max = max(r.max() for r in all_rms_db)
        rms_global_max = max(rms_global_max, file_rms_db_max)

    # --- Second pass: plot with uniform y-axis ---
    for i in range(num_files):
        static_kick, _ = librosa.load(f'dataset/static_kick/static_kick_{i:03d}.wav', sr=sample_rate)
        static_snare, _ = librosa.load(f'dataset/static_snare/static_snare_{i:03d}.wav', sr=sample_rate)
        dynamic_kick, _ = librosa.load(f'dataset/dynamic_kick/dynamic_kick_{i:03d}.wav', sr=sample_rate)
        dynamic_snare, _ = librosa.load(f'dataset/dynamic_snare/dynamic_snare_{i:03d}.wav', sr=sample_rate)

        stems = [
            ('Static Kick', static_kick),
            ('Static Snare', static_snare),
            ('Dynamic Kick', dynamic_kick),
            ('Dynamic Snare', dynamic_snare),
        ]

        fig, axes = plt.subplots(4, 3, figsize=(18, 16))
        fig.suptitle(f'Stem Analysis — File {i:03d}', fontsize=16, fontweight='bold')

        freqs = librosa.fft_frequencies(sr=sample_rate, n_fft=FRAME)

        for row, (label, audio) in enumerate(stems):
            # Compute STFT
            stft = librosa.stft(audio, n_fft=FRAME, hop_length=HOP)
            mag = np.abs(stft)
            mag_db = librosa.amplitude_to_db(mag, ref=np.max)

            # --- Spectrogram ---
            img = librosa.display.specshow(
                mag_db, sr=sample_rate, hop_length=HOP, n_fft=FRAME,
                x_axis='time', y_axis='hz', ax=axes[row, 0], cmap='magma'
            )
            axes[row, 0].set_title(f'{label} — Spectrogram')
            axes[row, 0].set_ylabel('Frequency (Hz)')
            axes[row, 0].set_xlabel('Time (s)')
            fig.colorbar(img, ax=axes[row, 0], format='%+2.0f dB')

            # --- Average magnitude spectrum (W-like) in dB ---
            avg_mag = mag.mean(axis=1)
            avg_mag_db = 20 * np.log10(avg_mag + 1e-6)
            axes[row, 1].plot(freqs, avg_mag_db, color='tab:blue')
            axes[row, 1].set_title(f'{label} — Avg Magnitude Spectrum')
            axes[row, 1].set_xlabel('Frequency (Hz)')
            axes[row, 1].set_ylabel('Magnitude (dB)')
            axes[row, 1].set_xlim([0, sample_rate / 2])
            axes[row, 1].set_ylim([-120, 30])

            # --- Activation envelope (RMS over time, H-like) in dB ---
            times = librosa.frames_to_time(np.arange(mag.shape[1]), sr=sample_rate, hop_length=HOP)
            rms = np.sqrt(np.mean(mag ** 2, axis=0))
            rms_db = 20 * np.log10(rms + 1e-6)
            axes[row, 2].plot(times, rms_db, color='tab:orange')
            axes[row, 2].set_title(f'{label} — Activation (RMS Envelope)')
            axes[row, 2].set_xlabel('Time (s)')
            axes[row, 2].set_ylabel('RMS Magnitude (dB)')
            axes[row, 2].set_ylim([-120, 30])

        plt.tight_layout()
        plt.savefig(f'original_data_plot/stem_spectrogram_analysis_{i:03d}.png', dpi=150, bbox_inches='tight')
        plt.close()
        print(f'Saved stem_spectrogram_analysis_{i:03d}.png')

    print(f'Global RMS max = {rms_global_max:.4f}')
    return rms_global_max


# Call the function
rms_global_max = plot_stem_spectrograms(num_files, sample_rate, FRAME, HOP)

Saved stem_spectrogram_analysis_000.png
Global RMS max = 20.1450


### Run NMF

In [88]:
# Run NMF on static and dynamic mixtures — produce audio stems and plots
os.makedirs('nmf_results_plot', exist_ok=True)

def plot_nmf_static_dynamic(s_comp_specs, d_comp_specs,
                            file_idx, sr=22050, hop_length=256, n_fft=512,
                            save_path=None, h_ylim=None):
    """Plot Static NMF and Dynamic NMF component spectrograms in one figure.

    Each component is shown via its average magnitude spectrum (left) and
    RMS activation envelope (right), computed from the soft-masked,
    original-scale component spectrogram — directly comparable to the
    original stem analysis plots.

    Parameters
    ----------
    s_comp_specs, d_comp_specs : list of np.ndarray, each shape (numBins, numFrames)
        Soft-masked component magnitude spectrograms in original scale.
    h_ylim : tuple or None
        If provided, set the y-axis limits (min, max) for all activation subplots (in dB).
    """
    numComp = len(s_comp_specs)
    nrows = 2 * numComp
    fig, axes = plt.subplots(nrows, 2, figsize=(16, 3 * nrows))
    fig.suptitle(f'File {file_idx:03d} — NMF Decompositions', fontsize=16, fontweight='bold', y=1.0)
    freqs = librosa.fft_frequencies(sr=sr, n_fft=n_fft)

    def _plot_nmf(comp_specs, row_start, label):
        for r in range(numComp):
            row = row_start + r
            mag = comp_specs[r]
            # Average magnitude spectrum (same metric as original stem plots)
            avg_mag = mag.mean(axis=1)
            avg_mag_db = 20 * np.log10(avg_mag + 1e-6)
            axes[row, 0].plot(freqs, avg_mag_db)
            axes[row, 0].set_title(f'{label} — Avg Magnitude Spectrum (Component {r})')
            axes[row, 0].set_xlabel('Frequency (Hz)')
            axes[row, 0].set_ylabel('Magnitude (dB)')
            axes[row, 0].set_xlim([0, sr / 2])
            axes[row, 0].set_ylim([-120, 30])

            # RMS activation envelope (same metric as original stem plots)
            times = librosa.frames_to_time(np.arange(mag.shape[1]), sr=sr, hop_length=hop_length)
            rms = np.sqrt(np.mean(mag ** 2, axis=0))
            rms_db = 20 * np.log10(rms + 1e-6)
            axes[row, 1].plot(times, rms_db, color='tab:orange')
            axes[row, 1].set_title(f'{label} — Activation (RMS Envelope, Component {r})')
            axes[row, 1].set_xlabel('Time (s)')
            axes[row, 1].set_ylabel('RMS Magnitude (dB)')
            if h_ylim is not None:
                axes[row, 1].set_ylim(h_ylim)

    _plot_nmf(s_comp_specs, 0, 'Static NMF')
    _plot_nmf(d_comp_specs, numComp, 'Dynamic NMF')

    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.close(fig)
    else:
        plt.show()

for i in range(num_files):
    print(f"Processing file {i:03d} — NMF...")

    # --- Static NMF ---
    static_mix, _ = librosa.load(f'dataset/static/static_mix_{i:03d}.wav', sr=sample_rate)
    static_mix_stft = librosa.stft(static_mix, n_fft=FRAME, hop_length=HOP)
    V_static = np.abs(static_mix_stft) + EPS

    NMFparam_s = {
        'numComp': 2, 'numIter': 80,
        'numBins': V_static.shape[0], 'numFrames': V_static.shape[1],
        'initW': np.random.rand(V_static.shape[0], 2),
        'initH': np.random.rand(2, V_static.shape[1]),
        'costFunc': 'KLDiv', 'fixW': False,
    }

    s_W_nmf, s_H_nmf, nmfV_static = NMF(V_static.copy(), parameter=NMFparam_s)

    # Soft masking: use NMF components as ratio masks on the original spectrogram
    static_mix_phase = np.angle(static_mix_stft)
    static_V_orig = np.abs(static_mix_stft)
    approx_static = sum(nmfV_static) + EPS
    static_comp_specs = []
    for j, comp_mag in enumerate(nmfV_static):
        mask = comp_mag / approx_static
        comp_mag_rescaled = mask * static_V_orig
        static_comp_specs.append(comp_mag_rescaled)
        comp_spec = comp_mag_rescaled * np.exp(1j * static_mix_phase)
        y_j = librosa.istft(comp_spec, hop_length=HOP, length=len(static_mix))
        if j == 0:
            sf.write(f"NMFresults/static_kick/static_mix_{i:03d}_kick.wav", y_j, sample_rate)
        else:
            sf.write(f"NMFresults/static_snare/static_mix_{i:03d}_snare.wav", y_j, sample_rate)

    # --- Dynamic NMF ---
    dynamic_mix, _ = librosa.load(f'dataset/dynamic/dynamic_mix_{i:03d}.wav', sr=sample_rate)
    dynamic_mix_stft = librosa.stft(dynamic_mix, n_fft=FRAME, hop_length=HOP)
    V_dynamic = np.abs(dynamic_mix_stft) + EPS

    NMFparam_d = {
        'numComp': 2, 'numIter': 80,
        'numBins': V_dynamic.shape[0], 'numFrames': V_dynamic.shape[1],
        'initW': np.random.rand(V_dynamic.shape[0], 2),
        'initH': np.random.rand(2, V_dynamic.shape[1]),
        'costFunc': 'KLDiv', 'fixW': False,
    }

    d_W_nmf, d_H_nmf, nmfV_dynamic = NMF(V_dynamic.copy(), parameter=NMFparam_d)

    # Soft masking: use NMF components as ratio masks on the original spectrogram
    dynamic_mix_phase = np.angle(dynamic_mix_stft)
    dynamic_V_orig = np.abs(dynamic_mix_stft)
    approx_dynamic = sum(nmfV_dynamic) + EPS
    dynamic_comp_specs = []
    for j, comp_mag in enumerate(nmfV_dynamic):
        mask = comp_mag / approx_dynamic
        comp_mag_rescaled = mask * dynamic_V_orig
        dynamic_comp_specs.append(comp_mag_rescaled)
        comp_spec = comp_mag_rescaled * np.exp(1j * dynamic_mix_phase)
        y_j = librosa.istft(comp_spec, hop_length=HOP, length=len(dynamic_mix))
        if j == 0:
            sf.write(f"NMFresults/dynamic_kick/dynamic_mix_{i:03d}_kick.wav", y_j, sample_rate)
        else:
            sf.write(f"NMFresults/dynamic_snare/dynamic_mix_{i:03d}_snare.wav", y_j, sample_rate)

    # --- Plot NMF decompositions (rescaled component spectrograms) ---
    h_ylim_db = (-120, 30)
    plot_nmf_static_dynamic(
        static_comp_specs, dynamic_comp_specs,
        file_idx=i, sr=sample_rate, hop_length=HOP, n_fft=FRAME,
        save_path=f"nmf_results_plot/nmf_decomposition_{i:03d}.png",
        h_ylim=h_ylim_db,
    )

print("Done — NMF audio stems + plots saved")

Processing file 000 — NMF...


Processing:   0%|          | 0/80 [00:00<?, ?it/s]

Processing:   0%|          | 0/80 [00:00<?, ?it/s]

Done — NMF audio stems + plots saved


### Run NMFD

In [89]:
# Run NMFD on static and dynamic mixtures — produce audio stems and plots

def plot_nmfd_static_dynamic(s_comp_specs, d_comp_specs,
                             s_W, d_W,
                             file_idx, sr=22050, hop_length=256, n_fft=512,
                             save_path=None, h_ylim=None):
    """Plot Static NMFD and Dynamic NMFD component spectrograms in one figure.

    Each component is shown as a full spectrogram (left), its learned W
    template frames spectrogram (centre), and RMS activation envelope (right),
    computed from the soft-masked, original-scale component spectrogram —
    directly comparable to the original stem analysis plots.

    Parameters
    ----------
    s_comp_specs, d_comp_specs : list of np.ndarray, each shape (numBins, numFrames)
        Soft-masked component magnitude spectrograms in original scale.
    s_W, d_W : list of np.ndarray, each shape (numBins, numTemplateFrames)
        Learned NMFD template matrices for static / dynamic mixtures.
    h_ylim : tuple or None
        If provided, set the y-axis limits (min, max) for all activation subplots (in dB).
    """
    numComp = len(s_comp_specs)
    nrows = 2 * numComp
    fig, axes = plt.subplots(nrows, 3, figsize=(20, 4 * nrows))
    fig.suptitle(f'File {file_idx:03d} — NMFD Decompositions', fontsize=16, fontweight='bold', y=1.0)

    def _plot_nmfd(comp_specs, W_list, row_start, label):
        for r in range(numComp):
            row = row_start + r
            mag = comp_specs[r]
            mag_db = librosa.amplitude_to_db(mag, ref=np.max)

            # --- Component Spectrogram ---
            img = librosa.display.specshow(
                mag_db, sr=sr, hop_length=hop_length, n_fft=n_fft,
                x_axis='time', y_axis='hz', ax=axes[row, 0], cmap='magma'
            )
            axes[row, 0].set_title(f'{label} — Spectrogram (Component {r})')
            axes[row, 0].set_ylabel('Frequency (Hz)')
            axes[row, 0].set_xlabel('Time (s)')
            fig.colorbar(img, ax=axes[row, 0], format='%+2.0f dB')

            # --- W Template Frames Spectrogram ---
            W_r = W_list[r]  # shape (numBins, numTemplateFrames)
            W_r_db = librosa.amplitude_to_db(W_r, ref=np.max)
            freqs = librosa.fft_frequencies(sr=sr, n_fft=n_fft)
            template_extent = [0, W_r.shape[1], freqs[0], freqs[-1]]
            img_w = axes[row, 1].imshow(
                W_r_db, aspect='auto', origin='lower',
                extent=template_extent, cmap='magma',
                vmin=-80, vmax=0
            )
            axes[row, 1].set_title(f'{label} — W Template Frames (Component {r})')
            axes[row, 1].set_ylabel('Frequency (Hz)')
            axes[row, 1].set_xlabel('Template Frame')
            fig.colorbar(img_w, ax=axes[row, 1], format='%+2.0f dB')

            # --- RMS activation envelope ---
            times = librosa.frames_to_time(np.arange(mag.shape[1]), sr=sr, hop_length=hop_length)
            rms = np.sqrt(np.mean(mag ** 2, axis=0))
            rms_db = 20 * np.log10(rms + 1e-6)
            axes[row, 2].plot(times, rms_db, color='tab:orange')
            axes[row, 2].set_title(f'{label} — Activation (RMS Envelope, Component {r})')
            axes[row, 2].set_xlabel('Time (s)')
            axes[row, 2].set_ylabel('RMS Magnitude (dB)')
            if h_ylim is not None:
                axes[row, 2].set_ylim(h_ylim)

    _plot_nmfd(s_comp_specs, s_W, 0, 'Static NMFD')
    _plot_nmfd(d_comp_specs, d_W, numComp, 'Dynamic NMFD')

    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.close(fig)
    else:
        plt.show()

for i in range(num_files):
    print(f"Processing file {i:03d} — NMFD...")

    # --- Static NMFD ---
    static_mix, _ = librosa.load(f'dataset/static/static_mix_{i:03d}.wav', sr=sample_rate)
    static_mix_stft = librosa.stft(static_mix, n_fft=FRAME, hop_length=HOP)
    V_static = np.abs(static_mix_stft) + EPS

    NMFDparam_s = {
        'numComp': 2, 'numIter': 80, 'numTemplateFrames': template_frames,
        'numBins': V_static.shape[0], 'numFrames': V_static.shape[1],
    }

    s_W_nmfd, s_H_nmfd, nmfdV_static, _, _ = NMFD(V_static.copy(), parameter=NMFDparam_s)

    # Soft masking: use NMFD components as ratio masks on the original spectrogram
    static_mix_phase = np.angle(static_mix_stft)
    static_V_orig = np.abs(static_mix_stft)
    approx_static = sum(nmfdV_static) + EPS
    static_comp_specs = []
    for j, comp_mag in enumerate(nmfdV_static):
        mask = comp_mag / approx_static
        comp_mag_rescaled = mask * static_V_orig
        static_comp_specs.append(comp_mag_rescaled)
        comp_spec = comp_mag_rescaled * np.exp(1j * static_mix_phase)
        y_j = librosa.istft(comp_spec, hop_length=HOP, length=len(static_mix))
        if j == 0:
            sf.write(f"NMFDresults/static_kick/static_mix_{i:03d}_kick.wav", y_j, sample_rate)
        else:
            sf.write(f"NMFDresults/static_snare/static_mix_{i:03d}_snare.wav", y_j, sample_rate)

    # --- Dynamic NMFD ---
    dynamic_mix, _ = librosa.load(f'dataset/dynamic/dynamic_mix_{i:03d}.wav', sr=sample_rate)
    dynamic_mix_stft = librosa.stft(dynamic_mix, n_fft=FRAME, hop_length=HOP)
    V_dynamic = np.abs(dynamic_mix_stft) + EPS

    NMFDparam_d = {
        'numComp': 2, 'numIter': 80, 'numTemplateFrames': template_frames,
        'numBins': V_dynamic.shape[0], 'numFrames': V_dynamic.shape[1],
    }

    d_W_nmfd, d_H_nmfd, nmfdV_dynamic, _, _ = NMFD(V_dynamic.copy(), parameter=NMFDparam_d)

    # Soft masking: use NMFD components as ratio masks on the original spectrogram
    dynamic_mix_phase = np.angle(dynamic_mix_stft)
    dynamic_V_orig = np.abs(dynamic_mix_stft)
    approx_dynamic = sum(nmfdV_dynamic) + EPS
    dynamic_comp_specs = []
    for j, comp_mag in enumerate(nmfdV_dynamic):
        mask = comp_mag / approx_dynamic
        comp_mag_rescaled = mask * dynamic_V_orig
        dynamic_comp_specs.append(comp_mag_rescaled)
        comp_spec = comp_mag_rescaled * np.exp(1j * dynamic_mix_phase)
        y_j = librosa.istft(comp_spec, hop_length=HOP, length=len(dynamic_mix))
        if j == 0:
            sf.write(f"NMFDresults/dynamic_kick/dynamic_mix_{i:03d}_kick.wav", y_j, sample_rate)
        else:
            sf.write(f"NMFDresults/dynamic_snare/dynamic_mix_{i:03d}_snare.wav", y_j, sample_rate)

    # --- Plot NMFD decompositions (rescaled component spectrograms) ---
    h_ylim_db = (-120, 30)
    plot_nmfd_static_dynamic(
        static_comp_specs, dynamic_comp_specs,
        s_W_nmfd, d_W_nmfd,
        file_idx=i, sr=sample_rate, hop_length=HOP, n_fft=FRAME,
        save_path=f"nmfd_results_plot/nmfd_decomposition_{i:03d}.png",
        h_ylim=h_ylim_db,
    )

print("Done — NMFD audio stems + plots saved")

Processing file 000 — NMFD...


Processing:   0%|          | 0/80 [00:00<?, ?it/s]

Processing:   0%|          | 0/80 [00:00<?, ?it/s]

Done — NMFD audio stems + plots saved


### Metrics

In [90]:
#calculate metrics
all_static_nmf_sdr = []
all_static_nmf_sir = []
all_static_nmf_sar = []
all_static_nmfd_sdr = []
all_static_nmfd_sir = []
all_static_nmfd_sar = []
all_dynamic_nmf_sdr = []
all_dynamic_nmf_sir = []
all_dynamic_nmf_sar = []
all_dynamic_nmfd_sdr = []
all_dynamic_nmfd_sir = []
all_dynamic_nmfd_sar = []

for i in range(num_files):
    # Load reference sources (ground truth)
    static_reference_snare, _ = librosa.load(f'dataset/static_snare/static_snare_{i:03d}.wav', sr=sample_rate)
    static_reference_kick, _ = librosa.load(f'dataset/static_kick/static_kick_{i:03d}.wav', sr=sample_rate)
    static_reference_sources = np.array([static_reference_kick, static_reference_snare])

    dynamic_reference_snare, _ = librosa.load(f'dataset/dynamic_snare/dynamic_snare_{i:03d}.wav', sr=sample_rate)
    dynamic_reference_kick, _ = librosa.load(f'dataset/dynamic_kick/dynamic_kick_{i:03d}.wav', sr=sample_rate)
    dynamic_reference_sources = np.array([dynamic_reference_kick, dynamic_reference_snare])

    # Load static NMF estimated sources
    static_nmf_kick, _ = librosa.load(f'NMFresults/static_kick/static_mix_{i:03d}_kick.wav', sr=sample_rate)
    static_nmf_snare, _ = librosa.load(f'NMFresults/static_snare/static_mix_{i:03d}_snare.wav', sr=sample_rate)
    static_nmf_estimated_sources = np.array([static_nmf_kick, static_nmf_snare])

    # Load static NMFD estimated sources
    static_nmfd_kick, _ = librosa.load(f'NMFDresults/static_kick/static_mix_{i:03d}_kick.wav', sr=sample_rate)
    static_nmfd_snare, _ = librosa.load(f'NMFDresults/static_snare/static_mix_{i:03d}_snare.wav', sr=sample_rate)
    static_nmfd_estimated_sources = np.array([static_nmfd_kick, static_nmfd_snare])

    # Load dynamic NMF estimated sources
    dynamic_nmf_kick, _ = librosa.load(f'NMFresults/dynamic_kick/dynamic_mix_{i:03d}_kick.wav', sr=sample_rate)
    dynamic_nmf_snare, _ = librosa.load(f'NMFresults/dynamic_snare/dynamic_mix_{i:03d}_snare.wav', sr=sample_rate)
    dynamic_nmf_estimated_sources = np.array([dynamic_nmf_kick, dynamic_nmf_snare])

    # Load dynamic NMFD estimated sources
    dynamic_nmfd_kick, _ = librosa.load(f'NMFDresults/dynamic_kick/dynamic_mix_{i:03d}_kick.wav', sr=sample_rate)
    dynamic_nmfd_snare, _ = librosa.load(f'NMFDresults/dynamic_snare/dynamic_mix_{i:03d}_snare.wav', sr=sample_rate)
    dynamic_nmfd_estimated_sources = np.array([dynamic_nmfd_kick, dynamic_nmfd_snare])

    # Ensure static arrays have the same length
    static_min_len = min(static_reference_sources.shape[1], static_nmf_estimated_sources.shape[1], static_nmfd_estimated_sources.shape[1])
    static_reference_sources = static_reference_sources[:, :static_min_len]
    static_nmf_estimated_sources = static_nmf_estimated_sources[:, :static_min_len]
    static_nmfd_estimated_sources = static_nmfd_estimated_sources[:, :static_min_len]

    # Ensure dynamic arrays have the same length
    dynamic_min_len = min(dynamic_reference_sources.shape[1], dynamic_nmf_estimated_sources.shape[1], dynamic_nmfd_estimated_sources.shape[1])
    dynamic_reference_sources = dynamic_reference_sources[:, :dynamic_min_len]
    dynamic_nmf_estimated_sources = dynamic_nmf_estimated_sources[:, :dynamic_min_len]
    dynamic_nmfd_estimated_sources = dynamic_nmfd_estimated_sources[:, :dynamic_min_len]

    # Evaluate static against static references
    # bss_eval_sources returns metrics already ordered by reference index [kick, snare]
    static_nmf_sdr, static_nmf_sir, static_nmf_sar, _ = bss_eval_sources(static_reference_sources, static_nmf_estimated_sources)
    static_nmfd_sdr, static_nmfd_sir, static_nmfd_sar, _ = bss_eval_sources(static_reference_sources, static_nmfd_estimated_sources)

    # Evaluate dynamic against dynamic references
    dynamic_nmf_sdr, dynamic_nmf_sir, dynamic_nmf_sar, _ = bss_eval_sources(dynamic_reference_sources, dynamic_nmf_estimated_sources)
    dynamic_nmfd_sdr, dynamic_nmfd_sir, dynamic_nmfd_sar, _ = bss_eval_sources(dynamic_reference_sources, dynamic_nmfd_estimated_sources)

    all_static_nmf_sdr.append(static_nmf_sdr)
    all_static_nmf_sir.append(static_nmf_sir)
    all_static_nmf_sar.append(static_nmf_sar)
    all_static_nmfd_sdr.append(static_nmfd_sdr)
    all_static_nmfd_sir.append(static_nmfd_sir)
    all_static_nmfd_sar.append(static_nmfd_sar)
    all_dynamic_nmf_sdr.append(dynamic_nmf_sdr)
    all_dynamic_nmf_sir.append(dynamic_nmf_sir)
    all_dynamic_nmf_sar.append(dynamic_nmf_sar)
    all_dynamic_nmfd_sdr.append(dynamic_nmfd_sdr)
    all_dynamic_nmfd_sir.append(dynamic_nmfd_sir)
    all_dynamic_nmfd_sar.append(dynamic_nmfd_sar)

    print(f"\n{i:03d} - Static NMF SDR: {static_nmf_sdr}")
    print(f"{i:03d} - Static NMF SIR: {static_nmf_sir}")
    print(f"{i:03d} - Static NMF SAR: {static_nmf_sar}")
    print(f"\n{i:03d} - Static NMFD SDR: {static_nmfd_sdr}")
    print(f"{i:03d} - Static NMFD SIR: {static_nmfd_sir}")
    print(f"{i:03d} - Static NMFD SAR: {static_nmfd_sar}")
    print(f"\n{i:03d} - Dynamic NMF SDR: {dynamic_nmf_sdr}")
    print(f"{i:03d} - Dynamic NMF SIR: {dynamic_nmf_sir}")
    print(f"{i:03d} - Dynamic NMF SAR: {dynamic_nmf_sar}")
    print(f"\n{i:03d} - Dynamic NMFD SDR: {dynamic_nmfd_sdr}")
    print(f"{i:03d} - Dynamic NMFD SIR: {dynamic_nmfd_sir}")
    print(f"{i:03d} - Dynamic NMFD SAR: {dynamic_nmfd_sar}")

/var/folders/zs/xdh0v3x55sz09h0zynkg0lz80000gq/T/ipykernel_6409/3619488180.py:59: FutureWarning: mir_eval.separation.bss_eval_sources
	Deprecated as of mir_eval version 0.8.
	It will be removed in mir_eval version 0.9.
  static_nmf_sdr, static_nmf_sir, static_nmf_sar, _ = bss_eval_sources(static_reference_sources, static_nmf_estimated_sources)
/var/folders/zs/xdh0v3x55sz09h0zynkg0lz80000gq/T/ipykernel_6409/3619488180.py:60: FutureWarning: mir_eval.separation.bss_eval_sources
	Deprecated as of mir_eval version 0.8.
	It will be removed in mir_eval version 0.9.
  static_nmfd_sdr, static_nmfd_sir, static_nmfd_sar, _ = bss_eval_sources(static_reference_sources, static_nmfd_estimated_sources)
/var/folders/zs/xdh0v3x55sz09h0zynkg0lz80000gq/T/ipykernel_6409/3619488180.py:63: FutureWarning: mir_eval.separation.bss_eval_sources
	Deprecated as of mir_eval version 0.8.
	It will be removed in mir_eval version 0.9.
  dynamic_nmf_sdr, dynamic_nmf_sir, dynamic_nmf_sar, _ = bss_eval_sources(dynamic_ref


000 - Static NMF SDR: [ 3.46903411 13.76903317]
000 - Static NMF SIR: [16.94004307 37.68969414]
000 - Static NMF SAR: [ 3.75583363 13.78741652]

000 - Static NMFD SDR: [ 3.57935997 13.90831359]
000 - Static NMFD SIR: [17.01834564 46.55083103]
000 - Static NMFD SAR: [ 3.86613412 13.9107737 ]

000 - Dynamic NMF SDR: [ 9.11239415 12.01730105]
000 - Dynamic NMF SIR: [11.9156743  40.99318591]
000 - Dynamic NMF SAR: [12.6108528  12.02314792]

000 - Dynamic NMFD SDR: [28.90866073 30.0384066 ]
000 - Dynamic NMFD SIR: [37.91739586 46.87191582]
000 - Dynamic NMFD SAR: [29.49245732 30.12948189]


In [91]:
#calculate metrics averages
static_nmf_sdr_average = sum(all_static_nmf_sdr) / len(all_static_nmf_sdr)
static_nmf_sir_average = sum(all_static_nmf_sir) / len(all_static_nmf_sir)
static_nmf_sar_average = sum(all_static_nmf_sar) / len(all_static_nmf_sar)

static_nmfd_sdr_average = sum(all_static_nmfd_sdr) / len(all_static_nmfd_sdr)
static_nmfd_sir_average = sum(all_static_nmfd_sir) / len(all_static_nmfd_sir)
static_nmfd_sar_average = sum(all_static_nmfd_sar) / len(all_static_nmfd_sar)

dynamic_nmf_sdr_average = sum(all_dynamic_nmf_sdr) / len(all_dynamic_nmf_sdr)
dynamic_nmf_sir_average = sum(all_dynamic_nmf_sir) / len(all_dynamic_nmf_sir)
dynamic_nmf_sar_average = sum(all_dynamic_nmf_sar) / len(all_dynamic_nmf_sar)

dynamic_nmfd_sdr_average = sum(all_dynamic_nmfd_sdr) / len(all_dynamic_nmfd_sdr)
dynamic_nmfd_sir_average = sum(all_dynamic_nmfd_sir) / len(all_dynamic_nmfd_sir)
dynamic_nmfd_sar_average = sum(all_dynamic_nmfd_sar) / len(all_dynamic_nmfd_sar)

print("Averages: \n")
print("Static NMF Average SDR:", static_nmf_sdr_average)
#print("Static NMF Average SIR:", static_nmf_sir_average)
#print("Static NMF Average SAR:", static_nmf_sar_average)
print("Static NMFD Average SDR:", static_nmfd_sdr_average)
#print("Static NMFD Average SIR:", static_nmfd_sir_average)
#print("Static NMFD Average SAR:", static_nmfd_sar_average)
print("Dynamic NMF Average SDR:", dynamic_nmf_sdr_average)
#print("Dynamic NMF Average SIR:", dynamic_nmf_sir_average)
#print("Dynamic NMF Average SAR:", dynamic_nmf_sar_average)
print("Dynamic NMFD Average SDR:", dynamic_nmfd_sdr_average)
#print("Dynamic NMFD Average SIR:", dynamic_nmfd_sir_average)
#print("Dynamic NMFD Average SAR:", dynamic_nmfd_sar_average)

Averages: 

Static NMF Average SDR: [ 3.46903411 13.76903317]
Static NMFD Average SDR: [ 3.57935997 13.90831359]
Dynamic NMF Average SDR: [ 9.11239415 12.01730105]
Dynamic NMFD Average SDR: [28.90866073 30.0384066 ]
